# Phase 4
## Model Training in Fraud Detection
We are training our model, which involves choosing the right Algorithms, understanding why they work and protecting against overfitting and systematically comparing candidates before picking a winner

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import(
    cross_val_score,
    StratifiedKFold,
    learning_curve
)
from sklearn.metrics import f1_score
import joblib
import os

# Load Phase 3 output
X_train = pd.read_csv('phase3_output/X_train_final.csv')
X_test  = pd.read_csv('phase3_output/X_test_final.csv')
y_train = pd.read_csv('phase3_output/y_train_final.csv').squeeze()
y_test  = pd.read_csv('phase3_output/y_test_final.csv').squeeze()

print(f"Training set: {X_train.shape[0]:,} rows, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Training fraud rate: {y_train.mean()*100:.1f}%  (post-SMOTE)")
print(f"Test fraud rate:     {y_test.mean()*100:.4f}%  (real world)")

In [ ]:
# Logistic Regression---Train model
print("Training Logistic Regression...")
start = time.time()

lr_model = LogisticRegression(
    C=0.1,                # Regularisation strength — smaller = more regularized
                          # Regularisation penalizes overly complex models
                          # C=0.1 means "be conservative, don't overfit"
    max_iter=1000,        # Max optimization steps before giving up
                          # Default 100 often fails to converge on large data
    solver='lbfgs',       # Optimisation algorithm — efficient for medium datasets
    class_weight=None,    # We already handled imbalance with SMOTE, so None here
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)

lr_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Training complete in {elapsed:.1f}s")

# Quick check on training data - how well does it fit
lr_train_preds = lr_model.predict(X_train)
lr_train_f1    = f1_score(y_train, lr_train_preds)
print(f"Training F1 score: {lr_train_f1:.4f}")
# This should be decent but not 1.0 — a perfect training score would mean overfit

In [ ]:
# Random Forest---Train model
print("Training Random Forest...")
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,      # Number of trees to build and vote
                           # More trees = more stable, but slower
                           # 100 is a reliable starting point

    max_depth=10,          # How deep each tree can grow
                           # Deeper = more complex patterns learned
                           # But also more risk of overfitting
                           # 10 is a good balance for tabular fraud data

    min_samples_leaf=4,    # Minimum rows required at a leaf node
                           # Prevents trees from fitting to single outliers
                           # Think of it as: "don't make a rule for just 1 person"

    max_features='sqrt',   # Each tree sees sqrt(30) ≈ 5 features at each split
                           # This forces diversity across trees — they learn
                           # different aspects of the data

    class_weight=None,     # SMOTE already balanced the data
    random_state=42,
    n_jobs=-1              # All CPU cores
)

rf_model.fit(X_train, y_train)

elapsed = time.time() - start
print(f"Traing complete in {elapsed:.1f}s")

rf_train_preds = rf_model.predict(X_train)
rf_train_f1    = f1_score(y_train, rf_train_preds)
print(f"Training F1 score: {rf_train_f1:.4f}")